In [ ]:
import pandas as pd
import duckdb
import os
import requests
from tqdm import tqdm 
import time

# Ingesting historical fpl data from vaastav (the GOAT)

In [ ]:
filepath = os.path.join(os.getcwd(), "data","FPL_historical.parquet")

fpl_seasons = ["2016-17", "2017-18", "2018-19", "2019-20", "2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

for s in fpl_seasons:
    url = f"https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/{s}/gws/merged_gw.csv"
    print(f"Processing Season: {s}")
    
    try:
        df = pd.read_csv(url, encoding='latin1')
        df['season'] = s
        df.to_parquet(filepath)
        
        
    except Exception as e:
        print(f"Error ingesting {s}: {e}")

#### The reason why I am ingesting data only from 2016-17 to 2024-25:
    - I want to add the column/feature 'code' (an unique id) for all the players from Chris Musson's FPL_id_map because vaastav has not added code in his dataset (in the merged gws)
    - And my weekly ingest and current season backfill maps the code from the bootstrapstatic while ingesting, so we don't need to worry about that later

### Bronze -> Silver

In [ ]:
file_path = os.path.join(os.getcwd(), "data/FPL_historical.parquet")
elementxcode_url = "https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/main/FPL/"

output_path = "data/silver/FPL_silver.parquet" ## Saving it in silver layer

seasons = ["16-17", "17-18", "18-19", "19-20", "20-21", "21-22", "22-23", "23-24", "24-25"]

all_ids = []

for season in seasons:
    full_season = f"20{season}" 
    try:
        url = f"{elementxcode_url}{season}.csv"
        df = pd.read_csv(url)
        if season in df.columns:
            df = df.rename(columns={season: 'element'})
        df['season'] = full_season
        all_ids.append(df)

    except Exception as e:
        print(f"Could not retrieve {season}: {e}")

data2 = pd.concat(all_ids, ignore_index=True)


query = f"""
    COPY (
            SELECT df2.code::VARCHAR AS code, 
            df1.*
            FROM read_parquet('{file_path}') AS df1
            LEFT JOIN data2 AS df2
            ON df1.element = df2.element
            AND df1.season = df2.season
        ) TO '{output_path}' (FORMAT 'PARQUET');
        """

duckdb.execute(query)

In [ ]:
# 2025-2026 season backfill and storing it in the silver layer
### Technically the medallion structure suggest that I append the data, along with the transformations in the silver but since its a one of script I'm doing it this way

boot_url = "https://fantasy.premierleague.com/api/bootstrap-static/"
boot_data = requests.get(boot_url).json()

team_map = {t['id']: t['name'] for t in boot_data['teams']}
pos_map = {1: 'GKP', 2: 'DEF', 3: 'MID', 4: 'FWD'}

player_map = {
    p['id']: {
        'name': f"{p['first_name']}_{p['second_name']}",
        'team': team_map.get(p['team']),
        'position': pos_map.get(p['element_type']),
        'xP': float(p['ep_next'] or 0),
        'code': str(p['code']) 
    }
    for p in boot_data['elements']
}

curr_season = []

for p_id in tqdm(player_map.keys(), desc="Backfilling fpl_25"):
    try:
        url = f"https://fantasy.premierleague.com/api/element-summary/{p_id}/"
        r = requests.get(url)
        if r.status_code == 200:
            data = r.json()
            if 'history' in data:
                for player in data['history']:
                    player.update(player_map[p_id])
                    player['season'] = '2025-26'
                    curr_season.append(player)
        
        time.sleep(0.4) # Respect the API!
    except Exception as e:
        print(f"Error on player {p_id}: {e}")

df_25 = pd.DataFrame(curr_season)
df_25['code'] = df_25['code'].astype(str)

In [ ]:
existing_df = pd.read_parquet("data/silver/FPL_silver.parquet")

combined_df = pd.concat([existing_df, df_25], join='outer', sort=False)

numeric_cols = [
'expected_goals', 'expected_assists', 'expected_goal_involvements', 
'expected_goals_conceded', 'value', 'selected', 'transfers_in','transfers_out', 
'influence', 'creativity', 'threat', 'ict_index', 'xp']

for col in numeric_cols:
    if col in combined_df.columns:
        combined_df[col] = (pd.to_numeric(combined_df[col], errors='coerce').fillna(0.0).astype('float64'))

combined_df.to_parquet("data/silver/FPL_silver.parquet", index=False)

### Silver layer transformations

#### FPl_silver x ustat (xG & xA)
##### I am only considering xG/xA, position and team name from udnerstat cause xG/xA was only introduced recently in the FPL but understat has data even before February 2023
##### Additionally, vaastav's data has a few null values for team and position. To achieve the goal of habing a "Single source of truth" and having a clean data, I ma doing the things I ma doing

#### Checklist before joining data
- [ ] Drop ghost columns and Columns where position is "AM"
- [ ] Map historical understat to fpl_silver
- [ ] Map ustat position and teams according to pos_map
- [ ] Calculate defensive contribution for seasons != 2025-26 based on ustat position
- [ ] Need to change the time format in fpl 
    (Automate in the weekly inside fpl x ustat join as a function )
- [ ] Map the understat ids to the players 
    (Need to automate in the weekly sync )

In [ ]:
fpl_silver_path = os.path.join(os.getcwd(), "data/silver/FPL_silver_cleaned.parquet")

combined_query = f"""
COPY (
    SELECT 
        * EXCLUDE (
            GW, mng_win,mng_loss, mng_draw, 
            mng_clean_sheets, mng_goals_scored, 
            mng_underdog_draw, mng_underdog_win, 
            defensive_contribution
        ),
        CASE 
            WHEN season = '2025-26' THEN defensive_contribution -- Keep the current 2025-26 def_cont value exactly as it is
            -- calculating def_cont for previous seasons
            WHEN position = 'DEF' AND (clearances_blocks_interceptions + tackles) >= 10 THEN 2
            WHEN position IN ('MID', 'FWD') AND (clearances_blocks_interceptions + tackles + recoveries) >= 12 THEN 2
            
            ELSE null
        END AS defensive_contribution
    FROM read_parquet('{fpl_silver_path}')
) TO '{fpl_silver_path}' (FORMAT 'PARQUET');
"""

duckdb.execute(combined_query)
print("Transformation Complete: Dropped dead columns and updated historical def_contribution.")

In [ ]:
pd.read_parquet("C:\Programing\Projects\Fantasy Premier League\data\silver\FPL_silver.parquet").head()